# Evaluate & Compare Models

Loads trained models and compares top-1/top-3 and confusion on a common test split.

In [ ]:
from pathlib import Path; import sys, numpy as np, torch
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
SRC=(Path('..')/'src').resolve(); sys.path.insert(0,str(SRC))
from stroke_recognizer.data import build_dataloaders, DEFAULT_LABELS_HEBREW
from stroke_recognizer.model import StrokeConvBiGRU
LABELS=DEFAULT_LABELS_HEBREW
DATA_ROOT=str((Path('..')/'data').resolve())


In [ ]:
# Data
train_loader, val_loader, test_loader = build_dataloaders(root=DATA_ROOT, label_names=LABELS, n_points=96, batch_size=64)
test_loader = test_loader or val_loader
len(test_loader.dataset)


In [ ]:
# Stroke Conv+BiGRU
def eval_stroke_conv(best_path):
    ckpt=torch.load(best_path,map_location='cpu'); m=StrokeConvBiGRU(in_channels=9,num_classes=len(LABELS))
    m.load_state_dict(ckpt['model']); m.eval()
    top1s=[]; y_all=[]; p_all=[]
    with torch.no_grad():
        for x,y in test_loader:
            logits=m(x); p=logits.argmax(1)
            top1s.append((p==y).float().mean().item()); y_all.append(y.numpy()); p_all.append(p.numpy())
    y=np.concatenate(y_all); p=np.concatenate(p_all); acc=(p==y).mean()
    cm=confusion_matrix(y,p,labels=list(range(len(LABELS))))
    return acc, cm

acc_conv, cm_conv = eval_stroke_conv('runs/stroke_conv_bigru_nb/best.pt')
acc_conv


In [ ]:
disp=ConfusionMatrixDisplay(cm_conv, display_labels=LABELS)
fig,ax=plt.subplots(figsize=(7,7)); disp.plot(ax=ax, xticks_rotation=90, cmap='Blues'); plt.tight_layout(); plt.show()
